In [1]:
from copy import copy
from math import sqrt

In [2]:
class Mesh:
   
    def __init__(self):
        self._knutepunktPosisjon = []
        self._staver = []        
   
    def __str__(self):
        return "Antall knutepunkt = " + str(self.antallKnutepunkt()) + "\n" + \
               "Knutepunktposisjoner = " + str(self._knutepunktPosisjon) + "\n" + \
               "Antall staver = " + str(self.antallStaver()) + "\n" + \
               "Staver = " + str(self._staver)          
                       
    # Antall knutepunkt i modellen    
    def antallKnutepunkt(self):
        return len(self._knutepunktPosisjon)
                   
    # Setter posisjonen til et knutepunkt        
    def settKnutepunktPosisjon(self, knutepunktIndex, nyPosisjon):
        self._knutepunktPosisjon[knutepunktIndex] = nyPosisjon
       
    # Legger til knutepunkt
    def leggTilKnutepunkt(self, posisjon):
        self._knutepunktPosisjon.append(posisjon)      

    # legg til liste med nye knutepunkt        
    def leggTilFlereKnutepunkt(self, posisjon):
        self._knutepunktPosisjon += posisjon
       
    # finner posisjonen til et knutepunkt
    def knutepunktPosisjon(self, knutepunktIndex):
        return self._knutepunktPosisjon[knutepunktIndex]
   
    # legger til en stav
    def leggTilStav(self, knutepunkter):
        self._staver.append(knutepunkter)

    # legger til en liste med flere staver        
    def leggTilFlereStaver(self, listeKnutepunkter):
        self._staver += listeKnutepunkter
       
    # Gir antall staver
    def antallStaver(self):
        return len(self._staver)
       
    # Endrer knutepunktene til en stav    
    def endreKnutepunktTilStav(self, stavIndex, knutepunkt):
        self._stav[stavIndex] = knutepunkt        
       
    # Gir knutepunktet til en stav
    def knutepunktTilStav(self, stavIndex):
        return self._staver[stavIndex]
   
    # Gir en liste med de stavene som er koblet til et knutepunkt
    def staverInnTilKnutepunkt(self, knutepunktIndex):
        return [ [stav for stav in self._staver if stav[0] == knutepunktIndex or stav[1] == knutepunktIndex]]
       

In [3]:
class MeshIter:
   
    def __init__(self, mesh):
        self._start = 0;
        self._mesh = mesh
        self._stop = self._mesh.antallStaver() - 1
        self._num = 0
       
    def __iter__(self):
        return self
       
    def __next__(self):
        if self._num > self._stop:
            raise StopIteration
        else:
            self._num += 1
            return self._mesh.knutepunktTilStav(self._num-1)

In [4]:
import numpy as np
       
def lagStivhetsmatrise(mesh, knutepunktFastX, knutepunktFastY):
    m = mesh.antallKnutepunkt()
    matrise = np.zeros((2*m, 2*m))
    n = 0
    for stav in MeshIter(mesh):
        x1, y1 = mesh.knutepunktPosisjon(stav[0])
        x2, y2 = mesh.knutepunktPosisjon(stav[1])
        dx = x2 - x1
        dy = y2 - y1
        d = sqrt(dx**2 + dy**2)
        cosverdi = dx / d
        sinverdi = dy / d
        matrise[2*stav[0]][n] = cosverdi
        matrise[2*stav[0]+1][n] = sinverdi
        matrise[2*stav[1]][n] = cosverdi
        matrise[2*stav[1]+1][n] = sinverdi
        n += 1
    for i in knutepunktFastX:
        matrise[2*i][n] = 1
        n += 1
    for i in knutepunktFastY:
        matrise[2*i+1][n] = 1
        n += 1
    return matrise


In [5]:
mesh = Mesh()
mesh.leggTilFlereKnutepunkt([[ 0. , 0.], [ 3.,  0.], [ 3.,  4.], [ 6. , 0.], [ 6. , 4.], [ 9.,  0.], [ 9. , 4.], [12.,  0.]])
mesh.leggTilFlereStaver([[0, 1], [0, 2], [1, 2], [1, 3], [2, 3], [2, 4], [3, 4], [3, 5], [3, 6], [4, 6], [5, 6], [5, 7], [6, 7]])

print(mesh)

L = lagStivhetsmatrise(mesh, [0], [0,7])

print(L)

F=np.zeros([2*mesh.antallKnutepunkt(),1])

F[3,0]=12
F[7,0]=20
F[11,0]=12

print(F)


strekk = np.linalg.solve(L,F)
print(strekk)

Antall knutepunkt = 8
Knutepunktposisjoner = [[0.0, 0.0], [3.0, 0.0], [3.0, 4.0], [6.0, 0.0], [6.0, 4.0], [9.0, 0.0], [9.0, 4.0], [12.0, 0.0]]
Antall staver = 13
Staver = [[0, 1], [0, 2], [1, 2], [1, 3], [2, 3], [2, 4], [3, 4], [3, 5], [3, 6], [4, 6], [5, 6], [5, 7], [6, 7]]
[[ 1.   0.6  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   1.
   0.   0. ]
 [ 0.   0.8  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   1.   0. ]
 [ 1.   0.   0.   1.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0. ]
 [ 0.   0.   1.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0. ]
 [ 0.   0.6  0.   0.   0.6  1.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0. ]
 [ 0.   0.8  1.   0.  -0.8  0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0. ]
 [ 0.   0.   0.   1.   0.6  0.   0.   1.   0.6  0.   0.   0.   0.   0.
   0.   0. ]
 [ 0.   0.   0.   0.  -0.8  0.   1.   0.   0.8  0.   0.   0.   0.   0.
   0.   0. ]
 [ 0.   0.   0.   0.   0.   1.   0.   0.   0.   1.  